## ML Classical Baselines

In [1]:
import importlib
from pathlib import Path

from IPython.display import display

import utils.graphs as graphs
import utils.ml_pipeline as ml_pipeline
import utils.plots as ml_plots

importlib.reload(graphs)
importlib.reload(ml_pipeline)
importlib.reload(ml_plots)

from utils.cache import get_cache_path, get_results_path
from utils.csi_preprocessing import process_magnitude_data
from utils.ml_pipeline import (
    best_confusion_predictions,
    load_all_predictions,
    load_feature_dataframes,
    load_lovo_summary_tables,
    load_or_build_none_reference_features,
    load_params_lookup,
    load_raw_csi_data,
    lovo_aggregated_analysis_table,
    master_results_table,
    per_room_position_accuracy_table,
    plot_block_vs_lovo_position_accuracy,
    plot_lovo_fold_spread,
    print_normalization_discriminability,
    run_global_baselines,
    run_optional_grid_search,
    save_analysis_tables,
    save_lovo_analysis_table,
)
from utils.plots import (
    plot_band_error_cdf,
    plot_floor_plan_heatmap,
    plot_global_position_confusion_matrix,
    plot_localization_error_cdf_by_model,
    plot_model_band_error_boxplot,
    plot_position_confusion_by_true_room,
)

#### Project Configurations

In [2]:
PROJECT_ROOT = Path(r"C:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project")
DATA_DIR = PROJECT_ROOT / "data"
CALIBRATION_MODE = "rssi"   # ("none", "packet_norm", "rssi")

CSV_PROCESSING_OPTIONS = {
    "max_workers": 1,
    "cache_dir": None,
    "use_cache": True,
    "force_reprocess": False,
    "min_rssi_dbm": -95.0,
    "calibration_eps": 1e-12,
}

MAGNITUDE_PROCESSING_OPTIONS = {
    "normalization": "empty_baseline",  # none | zscore | minmax | packet_minmax | empty_baseline
    "epsilon": 1e-8,
}
NORMALIZATION_BASELINE_SCOPE = "per_session"  # per_session | per_user | global (empty_baseline only)
SHOW_MAGNITUDE_PLOT = False   # True -> interactively plot raw vs normalized CSI

FEATURE_EXTRACTION_OPTIONS = {
    "window_size": 60,
    "overlap_size": 30,
    "require_all_esps": False,
}

# MODELS_TO_RUN = ("RF", "KNN", "SVM")
MODELS_TO_RUN = ("RF", )
BANDS_TO_RUN = ("2.4 GHz", "5 GHz", "Fusion")
SPLIT_MODES = ("cross_session",)  # session 1 is frozen training; session 2 is evaluation only.
RUN_GRID_SEARCH = False
FORCE_RETRAIN = False
SAVE_PREDICTIONS = True
SVM_KERNEL = "rbf"            # "rbf" | "linear"
N_JOBS = -1

BLOCK_COUNT = 10
TEST_SIZE = 0.30
RANDOM_STATE = 42
ROW_SPACING = 1.0
COLUMN_SPACING = 1.0
SVM_FUSION_FALLBACK_SECONDS = 30 * 60

CONFUSION_DATASET = "Fusion"
CONFUSION_MODEL = "best"      # "best", "RF", "KNN", or "SVM"

SHOW_CDF_BY_BAND = True
SHOW_CDF_BY_MODEL = True
SHOW_BOXPLOT = True
SHOW_FLOOR_PLAN = True
SHOW_CONFUSION_MATRICES = True
SHOW_PER_ROOM_PLOTS = False

preproc_opts = dict(MAGNITUDE_PROCESSING_OPTIONS)
if preproc_opts.get("normalization") == "empty_baseline":
    preproc_opts["baseline_scope"] = NORMALIZATION_BASELINE_SCOPE
feat_opts = dict(FEATURE_EXTRACTION_OPTIONS)
feature_cache_dir = get_cache_path(preproc_opts, feat_opts)
results_dir = get_results_path(preproc_opts, feat_opts)
print(f"Feature cache path: {feature_cache_dir}")
print(f"Results path: {results_dir}")
summary_dir = results_dir / "summary"
tables_dir = results_dir / "tables"
tuning_dir = results_dir / "tuning"
plots_dir = results_dir / "plots"
for directory in (summary_dir, tables_dir, tuning_dir, plots_dir):
    directory.mkdir(parents=True, exist_ok=True)


def _slugify(value: str) -> str:
    return value.lower().replace(".", "-").replace(" ", "-").strip("-")

Feature cache path: c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\.cache\dataframes\preproc=norm-empty_baseline_scope-per_session\feat=win60-step30
Results path: c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\results\preproc=norm-empty_baseline_scope-per_session\feat=win60-step30


## Raw Data

In [3]:
magnitude_data, csv_diagnostics = load_raw_csi_data(
    DATA_DIR,
    calibration_mode=CALIBRATION_MODE,
    csv_options=CSV_PROCESSING_OPTIONS,
)

Scenarios present: 1
Locations: 53 | Users: 6 | ESPs: 19


In [4]:
if SHOW_MAGNITUDE_PLOT:
    processed_magnitude_data, _ = process_magnitude_data(magnitude_data, **preproc_opts)
    print("=== RAW magnitudes (before normalization) ===")
    graphs.plot_magnitude_analysis_interactive(magnitude_data)
    print(
        "=== NORMALIZED magnitudes (after normalization: "
        f"{preproc_opts.get('normalization', 'none')}) ==="
    )
    graphs.plot_magnitude_analysis_interactive(processed_magnitude_data)
    del processed_magnitude_data

#### Feature Dataframes

In [5]:
feature_dataframes = load_feature_dataframes(
    magnitude_data,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
    bands_to_run=BANDS_TO_RUN,
)

none_reference_dataframes = (
    feature_dataframes
    if preproc_opts.get("normalization") == "none"
    else load_or_build_none_reference_features(
        magnitude_data,
        active_preproc_opts=preproc_opts,
        feat_opts=feat_opts,
    )
)
fisher_diagnostics = print_normalization_discriminability(
    feature_dataframes,
    normalization=preproc_opts.get("normalization", "none"),
    reference_feature_dataframes=none_reference_dataframes,
    bands_to_run=BANDS_TO_RUN,
)
display(fisher_diagnostics)
del magnitude_data


[features] resolved cache path: c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\.cache\dataframes\preproc=norm-empty_baseline_scope-per_session\feat=win60-step30
[cache miss] c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\.cache\dataframes\preproc=norm-empty_baseline_scope-per_session\feat=win60-step30, computing...
[Z-0 inventory]
  total Z-0 files found: 171
  Z-0 count per (user, trial):
    (user=01, trial=01): 19
    (user=01, trial=02): 19
    (user=02, trial=01): 19
    (user=03, trial=01): 19
    (user=03, trial=02): 19
    (user=04, trial=01): 19
    (user=05, trial=01): 19
    (user=05, trial=02): 19
    (user=06, trial=01): 19
  breakdown by user / esp / trial:
    user=01 esp=01 trial=01: 1
    user=01 esp=01 trial=02: 1
    user=01 esp=02 trial=01: 1
    user=01 esp=02 trial=02: 1
    user=01 esp=03 trial=01: 1
    user=01 esp=03 trial=02: 1
    user=01 esp=04 trial=01: 1
    user=01 esp

,dataset,normalization,median_fisher_ratio,none_median_fisher_ratio
0,2.4 GHz,empty_baseline,0.069335,0.039848
1,5 GHz,empty_baseline,0.067332,0.019588
2,Fusion,empty_baseline,0.068953,0.026314


#### Model Parameters

In [6]:
TUNED_SUMMARY_PATH = tuning_dir / "tuned_summary.csv"
GRID_LOG_PATH = tuning_dir / "full_grid_log.csv"

params_lookup = load_params_lookup(
    TUNED_SUMMARY_PATH,
    models_to_run=MODELS_TO_RUN,
    bands_to_run=BANDS_TO_RUN,
    svm_kernel=SVM_KERNEL,
)
for key, value in params_lookup.items():
    print(f"{key}: {value}")

('RF', '2.4 GHz'): {'n_estimators': 500, 'max_features': 'log2', 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
('RF', '5 GHz'): {'n_estimators': 500, 'max_features': 'sqrt', 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
('RF', 'Fusion'): {'n_estimators': 500, 'max_features': 'log2', 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}


##### Optional Grid Search

In [7]:
grid_ran = run_optional_grid_search(
    feature_dataframes,
    run_grid_search=RUN_GRID_SEARCH,
    tuned_summary_path=TUNED_SUMMARY_PATH,
    grid_log_path=GRID_LOG_PATH,
    models_to_run=MODELS_TO_RUN,
    bands_to_run=BANDS_TO_RUN,
    svm_kernel=SVM_KERNEL,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    n_blocks=BLOCK_COUNT,
    n_jobs=N_JOBS,
    row_spacing=ROW_SPACING,
    column_spacing=COLUMN_SPACING,
)
if grid_ran:
    raise SystemExit("RUN_GRID_SEARCH=True completed; set it to False before running experiments.")

#### Global 52-Class Baselines

In [8]:
global_summary, global_predictions_by_key = run_global_baselines(
    feature_dataframes,
    params_lookup=params_lookup,
    models_to_run=MODELS_TO_RUN,
    bands_to_run=BANDS_TO_RUN,
    split_modes=SPLIT_MODES,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    n_blocks=BLOCK_COUNT,
    n_jobs=N_JOBS,
    results_dir=results_dir,
    summary_dir=summary_dir,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
    force_retrain=FORCE_RETRAIN,
    save_predictions=SAVE_PREDICTIONS,
    svm_fallback_seconds=SVM_FUSION_FALLBACK_SECONDS,
    row_spacing=ROW_SPACING,
    column_spacing=COLUMN_SPACING,
)
display(global_summary)

[cross_session] discovered trial-02 users: 01, 03, 05
[cross_session] n_train=17644 users=['01', '02', '03', '04', '05', '06'] positions=52
[cross_session] n_test=9269 users=['01', '03', '05'] positions=52
[cross_session] assertion passed: no trial-02 window is in training
[cross_session] all test positions are represented in training
[predictions] Saved c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\results\preproc=norm-empty_baseline_scope-per_session\feat=win60-step30\predictions\2_4ghz__rf__cross_session.parquet
[baseline] 2.4 GHz / RF / cross_session: majority_position_accuracy=0.0194, majority_room_accuracy=0.6537
[cross_session] discovered trial-02 users: 01, 03, 05
[cross_session] n_train=19238 users=['01', '02', '03', '04', '05', '06'] positions=52
[cross_session] n_test=9507 users=['01', '03', '05'] positions=52
[cross_session] assertion passed: no trial-02 window is in training
[cross_session] all test positions are represented in 

,dataset,model,split,n_estimators,max_features,max_depth,min_samples_split,min_samples_leaf,position_accuracy,macro_f1,...,median_distance_error,rmse_distance_error,p90_distance_error,samples,majority_position_accuracy,majority_room_accuracy,fit_seconds,predict_seconds,wall_seconds,used_estimator
0,2.4 GHz,RF,cross_session,500,log2,None,2,1,0.115223,0.072915,...,3.605551,4.867873,8.246211,9269.0,0.019420,0.653684,65.489247,0.802615,66.663634,RandomForestClassifier
1,5 GHz,RF,cross_session,500,sqrt,None,2,1,0.129484,0.097513,...,3.605551,4.575868,7.211103,9507.0,0.019459,0.655727,440.980603,1.023269,442.431556,RandomForestClassifier
2,Fusion,RF,cross_session,500,log2,None,2,1,0.164292,0.117892,...,3.162278,4.694097,8.062258,9264.0,0.019430,0.653821,100.274508,0.991170,101.698526,RandomForestClassifier


#### Analysis Tables

In [9]:
all_global_predictions = load_all_predictions(
    results_dir,
    models_to_run=MODELS_TO_RUN,
    bands_to_run=BANDS_TO_RUN,
    split_modes=SPLIT_MODES,
)

master_table = master_results_table(
    all_global_predictions,
    summary_path=summary_dir / "global_summary.csv",
)
per_room_table = per_room_position_accuracy_table(all_global_predictions)
save_analysis_tables(master_table, per_room_table, tables_dir=tables_dir)

display(master_table)
display(per_room_table)

[predictions cache hit] c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\results\preproc=norm-empty_baseline_scope-per_session\feat=win60-step30\predictions\2_4ghz__rf__cross_session.parquet
[predictions cache hit] c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\results\preproc=norm-empty_baseline_scope-per_session\feat=win60-step30\predictions\5ghz__rf__cross_session.parquet
[predictions cache hit] c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\results\preproc=norm-empty_baseline_scope-per_session\feat=win60-step30\predictions\fusion__rf__cross_session.parquet


ImportError: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).

#### LOVO cross-user analysis


In [ ]:
if "lovo" in SPLIT_MODES:
    lovo_per_fold, lovo_summary = load_lovo_summary_tables(summary_dir)
    lovo_table = lovo_aggregated_analysis_table(lovo_summary)
    save_lovo_analysis_table(lovo_table, tables_dir=tables_dir)

    plot_lovo_fold_spread(
        lovo_per_fold,
        bands=BANDS_TO_RUN,
        model="RF",
        save_path=plots_dir / f"{_slugify('lovo rf fold spread')}.pdf",
    )
    plot_block_vs_lovo_position_accuracy(
        globals().get("global_summary", master_table),
        lovo_summary,
        bands=BANDS_TO_RUN,
        model="RF",
        save_path=plots_dir / f"{_slugify('block vs lovo rf position accuracy')}.pdf",
    )

    display(lovo_table)
    display(lovo_per_fold.loc[lovo_per_fold["model"] == "RF"])
else:
    print("LOVO analysis skipped because 'lovo' is not in SPLIT_MODES.")

#### Analysis Figures

In [ ]:
if SHOW_CDF_BY_BAND:
    for band in BANDS_TO_RUN:
        plot_localization_error_cdf_by_model(
            all_global_predictions,
            dataset=band,
            save_path=plots_dir / f"cdf_by_model_{_slugify(band)}.pdf",
        )

if SHOW_CDF_BY_MODEL:
    for model in MODELS_TO_RUN:
        model_predictions = all_global_predictions.loc[all_global_predictions["model"] == model]
        plot_band_error_cdf(
            model_predictions,
            model_label=model,
            split_modes=SPLIT_MODES,
            band_order=BANDS_TO_RUN,
            save_path=plots_dir,
        )

if SHOW_BOXPLOT:
    plot_model_band_error_boxplot(
        all_global_predictions,
        models=MODELS_TO_RUN,
        bands=BANDS_TO_RUN,
        save_path=plots_dir / "boxplot_model_band_distance_error.pdf",
    )

#### Confusion Matrix And Floor Plan

In [ ]:
confusion_model, confusion_predictions = best_confusion_predictions(
    all_global_predictions,
    master_table,
    dataset=CONFUSION_DATASET,
    model=CONFUSION_MODEL,
)
print(f"Confusion/floor-plan model: {confusion_model} on {CONFUSION_DATASET}")

if SHOW_FLOOR_PLAN:
    plot_floor_plan_heatmap(
        confusion_predictions,
        title=f"{CONFUSION_DATASET} / {confusion_model} localization heatmap",
        save_path=plots_dir / f"floor_plan_{_slugify(CONFUSION_DATASET)}_{_slugify(confusion_model)}.pdf",
    )

if SHOW_CONFUSION_MATRICES:
    plot_global_position_confusion_matrix(
        confusion_predictions,
        dataset=CONFUSION_DATASET,
        normalize="true",
        save_path=plots_dir / f"confusion_{_slugify(CONFUSION_DATASET)}_{_slugify(confusion_model)}.pdf",
    )
    if SHOW_PER_ROOM_PLOTS:
        room_plot_dir = plots_dir / f"confusion_by_room_{_slugify(CONFUSION_DATASET)}_{_slugify(confusion_model)}"
        plot_position_confusion_by_true_room(
            confusion_predictions,
            dataset=CONFUSION_DATASET,
            normalize="true",
            save_path=room_plot_dir,
        )